# Train Base Antenna Model From Scratch

This notebook is the clean training entry point for the whole solution. It reads the antenna simulation data, builds physics-informed features, trains the base classifier and frequency/bandwidth regressors, saves the model artifact, and generates ranked candidate outputs.

It supports both data formats found in this project:

- Newer 4-parameter data: `LG, WR, LR, WF -> FL, FU, BW`
- Older 3-parameter data: `LG, WR, WF -> FL, FU, BW`

For this folder's current dataset, the 4-parameter workflow is expected.

In [1]:
from pathlib import Path
import json
import os
import pickle
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/wideband-mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/wideband-cache")

import numpy as np
import pandas as pd

from copy import deepcopy
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    recall_score,
    roc_auc_score,
)

RANDOM_STATE = 42
TARGET_FL_MIN = 4.5
TARGET_FU_MAX = 9.0
MIN_RESONANT_RECALL = 0.93

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists() and (PROJECT_DIR.parent / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "UBW-data-final.xlsx"
MODEL_DIR = PROJECT_DIR / "model"
PRED_DIR = PROJECT_DIR / "predictions"
MODEL_DIR.mkdir(exist_ok=True)
PRED_DIR.mkdir(exist_ok=True)

LG_VALS = [4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5]
WR_VALS = [0.75, 2.25, 2.5, 7.5, 8.5, 9.25, 9.5, 10.0, 10.5, 10.75, 11.0, 11.25, 11.5, 11.75, 12.0, 12.25, 12.5, 12.75]
LR_VALS = [3.0, 3.3]
WF_VALS_4P = [round(x, 1) for x in np.arange(2.0, 5.0 + 0.001, 0.1)]
WF_VALS_3P = [round(x, 1) for x in np.arange(2.0, 3.0 + 0.001, 0.1)]

print("Project:", PROJECT_DIR)
print("Data:", DATA_PATH)
print("Model output:", MODEL_DIR)
print("Prediction output:", PRED_DIR)

Project: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files
Data: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/data/UBW-data-final.xlsx
Model output: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/model
Prediction output: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/predictions


## 1. Load And Normalize Data

The project currently has a newer 4-parameter dataset. The older `main-code.py` script used a 3-parameter format, so this notebook detects the shape and trains accordingly.

In [2]:
def load_training_data(path):
    raw = pd.read_excel(path, header=0)
    if raw.shape[1] >= 8:
        df = raw.iloc[:, :8].copy()
        df.columns = ["SrNo", "LG", "WR", "LR", "WF", "FL", "FU", "BW"]
        input_cols = ["LG", "WR", "LR", "WF"]
    elif raw.shape[1] >= 7:
        df = raw.iloc[:, :7].copy()
        df.columns = ["SrNo", "LG", "WR", "WF", "FL", "FU", "BW"]
        input_cols = ["LG", "WR", "WF"]
    else:
        raise ValueError(f"Expected at least 7 columns, found {raw.shape[1]}")

    numeric_cols = input_cols + ["FL", "FU", "BW"]
    df = df.dropna(subset=numeric_cols).copy()
    df[numeric_cols] = df[numeric_cols].astype(float)
    df = (
        df.sort_values("BW", ascending=False)
        .drop_duplicates(subset=input_cols, keep="first")
        .reset_index(drop=True)
    )
    df["is_resonant"] = (df["BW"] > 0).astype(int)
    df["in_target_window"] = ((df["BW"] > 0) & (df["FL"] >= TARGET_FL_MIN) & (df["FU"] <= TARGET_FU_MAX)).astype(int)
    return df, input_cols

df, INPUT_COLS = load_training_data(DATA_PATH)
HAS_LR = "LR" in INPUT_COLS
print("Input columns:", INPUT_COLS)
print("Rows:", len(df))
print("Resonant:", int(df.is_resonant.sum()))
print("Dead:", int((df.is_resonant == 0).sum()))
print("Target-window rows:", int(df.in_target_window.sum()))
display(df[INPUT_COLS + ["FL", "FU", "BW", "is_resonant", "in_target_window"]].head(10))

Input columns: ['LG', 'WR', 'LR', 'WF']
Rows: 2039
Resonant: 1615
Dead: 424
Target-window rows: 1615


,LG,WR,LR,WF,FL,FU,BW,is_resonant,in_target_window
0,5.5,12.5,3.0,2.8,5.30,8.97,3.67,1,1
1,5.5,12.0,3.0,4.2,5.15,8.80,3.65,1,1
2,5.5,12.0,3.3,4.4,5.13,8.78,3.65,1,1
3,5.5,12.0,3.3,4.5,5.13,8.76,3.63,1,1
4,5.5,12.0,3.0,3.9,5.24,8.81,3.57,1,1
5,5.5,12.0,3.3,4.8,5.12,8.69,3.57,1,1
6,6.0,12.0,3.0,4.7,5.09,8.64,3.55,1,1
7,5.5,12.0,3.0,4.1,5.19,8.74,3.55,1,1
8,5.5,12.5,3.0,3.5,5.22,8.76,3.54,1,1
9,6.0,12.0,3.0,5.0,5.10,8.63,3.53,1,1


## 2. Physics-Informed Feature Builder

The feature set combines the newer `code/high_bw_resonant_model.py` 4-parameter features with a fallback path compatible with the older 3-parameter `main-code.py` workflow.

In [3]:
WR_B1, WR_B2, WR_B3, WR_B4, WR_B5 = 8.5, 10.75, 11.0, 11.25, 12.0
WF_PERIOD, LG_PERIOD = 0.55, 4.25

def build_features(df_in):
    LG = df_in["LG"].to_numpy(float)
    WR = df_in["WR"].to_numpy(float)
    WF = df_in["WF"].to_numpy(float)
    LR = df_in["LR"].to_numpy(float) if "LR" in df_in.columns else np.full(len(df_in), 3.0)

    wr_zone = np.where(
        WR < WR_B1,
        0,
        np.where(WR < WR_B2, 1, np.where(WR < WR_B3, 2, np.where(WR < WR_B4, 3, np.where(WR < WR_B5, 4, 5)))),
    ).astype(float)

    f = {
        "LG": LG,
        "WR": WR,
        "WF": WF,
        "LG2": LG**2,
        "WR2": WR**2,
        "WF2": WF**2,
        "LG3": LG**3,
        "WR3": WR**3,
        "WF3": WF**3,
        "LG_WR": LG * WR,
        "LG_WF": LG * WF,
        "WR_WF": WR * WF,
        "LG_WR_WF": LG * WR * WF,
        "LG2_WR": LG**2 * WR,
        "WR2_LG": WR**2 * LG,
        "WR_b1": (WR > WR_B1).astype(float),
        "WR_b2": (WR > WR_B2).astype(float),
        "WR_b3": (WR > WR_B3).astype(float),
        "WR_b4": (WR > WR_B4).astype(float),
        "WR_b5": (WR > WR_B5).astype(float),
        "WR_d11": WR - 11.0,
        "WR_d1125": np.clip(WR - 11.25, 0, None),
        "WR_zone": wr_zone,
        "zone_LG": wr_zone * LG,
        "logWR": np.log1p(WR),
        "logLG": np.log(LG),
        "logWF": np.log(WF),
        "WR_WFr": WR / (WF + 0.1),
        "LG_WRr": LG / (WR + 0.1),
        "WR_LGr": WR / (LG + 0.1),
        "WR_b3_LG_WF": (WR > WR_B3).astype(float) * LG * WF,
        "WR_d11_LG": (WR - 11.0) * LG,
        "WR_d11_WF": (WR - 11.0) * WF,
        "WF2_LG": WF**2 * LG,
        "LG_WF_zone": LG * WF * wr_zone,
    }

    if HAS_LR:
        f.update({
            "LR": LR,
            "LR2": LR**2,
            "LG_LR": LG * LR,
            "WR_LR": WR * LR,
            "LR_WF": LR * WF,
            "LG_WR_LR_WF": LG * WR * LR * WF,
            "zone_LR": wr_zone * LR,
            "LR_flag": (LR > 3.1).astype(float),
            "LR_delta": LR - 3.0,
            "LR_zone": (LR > 3.1).astype(float) * wr_zone,
            "LR_WR_d11": (LR - 3.0) * (WR - 11.0),
            "LR_WF2": LR * WF**2,
            "LR_LG_WF_zone": LR * LG * WF * wr_zone,
        })

    for k in [1, 2, 3]:
        angle = k * np.pi * WF / WF_PERIOD
        f[f"sinWF{k}"] = np.sin(angle)
        f[f"cosWF{k}"] = np.cos(angle)
    for k in [1, 2]:
        angle = k * np.pi * LG / LG_PERIOD
        f[f"sinLG{k}"] = np.sin(angle)
        f[f"cosLG{k}"] = np.cos(angle)
    if not HAS_LR:
        f["sinWF1_LG"] = np.sin(np.pi * WF / WF_PERIOD) * LG
        f["cosWF1_LG"] = np.cos(np.pi * WF / WF_PERIOD) * LG
    for boundary in [8.5, 10.75, 11.0, 11.25, 12.0, 12.5]:
        f[f"dWR_{boundary}"] = WR - boundary

    return pd.DataFrame(f)

X_all = build_features(df)
FEATURE_COLUMNS = X_all.columns.tolist()
print("Feature count:", len(FEATURE_COLUMNS))
display(X_all.head())

Feature count: 64


,LG,WR,WF,LG2,WR2,WF2,LG3,WR3,WF3,LG_WR,...,sinLG1,cosLG1,sinLG2,cosLG2,dWR_8.5,dWR_10.75,dWR_11.0,dWR_11.25,dWR_12.0,dWR_12.5
0,5.5,12.5,2.8,30.25,156.25,7.84,166.375,1953.125,21.952,68.75,...,-0.798017,-0.602635,0.961826,-0.273663,4.0,1.75,1.5,1.25,0.5,0.0
1,5.5,12.0,4.2,30.25,144.00,17.64,166.375,1728.000,74.088,66.00,...,-0.798017,-0.602635,0.961826,-0.273663,3.5,1.25,1.0,0.75,0.0,-0.5
2,5.5,12.0,4.4,30.25,144.00,19.36,166.375,1728.000,85.184,66.00,...,-0.798017,-0.602635,0.961826,-0.273663,3.5,1.25,1.0,0.75,0.0,-0.5
3,5.5,12.0,4.5,30.25,144.00,20.25,166.375,1728.000,91.125,66.00,...,-0.798017,-0.602635,0.961826,-0.273663,3.5,1.25,1.0,0.75,0.0,-0.5
4,5.5,12.0,3.9,30.25,144.00,15.21,166.375,1728.000,59.319,66.00,...,-0.798017,-0.602635,0.961826,-0.273663,3.5,1.25,1.0,0.75,0.0,-0.5


## 3. Train Classifier

The base model uses stacked tree ensembles. The classifier predicts whether a design is resonant (`BW > 0`). A logistic-regression meta-learner combines out-of-fold probabilities from the base classifiers.

In [4]:
def tune_resonant_threshold(y_true, prob, min_res_recall=MIN_RESONANT_RECALL):
    rows = []
    for threshold in np.arange(0.10, 0.91, 0.01):
        pred = (prob >= threshold).astype(int)
        resonant_recall = recall_score(y_true, pred, pos_label=1, zero_division=0)
        dead_recall = recall_score(y_true, pred, pos_label=0, zero_division=0)
        balanced_acc = balanced_accuracy_score(y_true, pred)
        f1 = f1_score(y_true, pred, zero_division=0)
        acc = accuracy_score(y_true, pred)
        penalty = max(0.0, min_res_recall - resonant_recall)
        objective = 0.45 * f1 + 0.25 * balanced_acc + 0.20 * resonant_recall + 0.10 * acc - 0.50 * penalty
        rows.append({
            "threshold": threshold,
            "accuracy": acc,
            "balanced_accuracy": balanced_acc,
            "f1": f1,
            "resonant_recall": resonant_recall,
            "dead_recall": dead_recall,
            "objective": objective,
        })
    table = pd.DataFrame(rows)
    feasible = table[table.resonant_recall >= min_res_recall]
    best = (feasible if len(feasible) else table).sort_values(["objective", "f1", "balanced_accuracy"], ascending=False).iloc[0]
    return float(best.threshold), table


def classifier_templates():
    return [
        ("ET", ExtraTreesClassifier(n_estimators=700, max_features="sqrt", min_samples_leaf=1, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ("RF", RandomForestClassifier(n_estimators=500, max_features="sqrt", min_samples_leaf=2, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)),
        ("HGB", HistGradientBoostingClassifier(max_iter=450, learning_rate=0.045, max_leaf_nodes=31, l2_regularization=0.02, class_weight="balanced", random_state=RANDOM_STATE)),
    ]


def fit_classifier(X_train, y_train):
    base = classifier_templates()
    min_class_count = np.bincount(y_train).min()
    n_folds = max(2, min(5, int(min_class_count)))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof = pd.DataFrame(index=np.arange(len(X_train)))

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train), 1):
        for name, model in base:
            fitted = deepcopy(model)
            fitted.fit(X_train.iloc[train_idx], y_train[train_idx])
            oof.loc[valid_idx, name] = fitted.predict_proba(X_train.iloc[valid_idx])[:, 1]
        print(f"classifier fold {fold}/{n_folds} done")

    meta = LogisticRegressionCV(Cs=20, cv=n_folds, class_weight="balanced", scoring="f1", max_iter=3000, random_state=RANDOM_STATE)
    meta.fit(oof, y_train)
    oof_prob = meta.predict_proba(oof)[:, 1]
    threshold, threshold_table = tune_resonant_threshold(y_train, oof_prob)

    final_base = []
    for name, model in base:
        fitted = deepcopy(model)
        fitted.fit(X_train, y_train)
        final_base.append((name, fitted))

    return {
        "base": final_base,
        "meta": meta,
        "threshold": threshold,
        "oof_prob": oof_prob,
        "threshold_table": threshold_table,
    }


def predict_res_prob(classifier, X_eval):
    base_prob = pd.DataFrame({name: model.predict_proba(X_eval)[:, 1] for name, model in classifier["base"]})
    return classifier["meta"].predict_proba(base_prob)[:, 1]

## 4. Train Regressors

Separate weighted ensembles are trained for `BW`, `FL`, and `FU` on resonant rows only. For 4-parameter data, rows inside the 4.5-9 GHz operating window are preferred.

In [5]:
REGRESSOR_WEIGHTS = np.array([0.40, 0.35, 0.25])

def regressor_templates():
    return [
        ("ET", ExtraTreesRegressor(n_estimators=800, max_features="sqrt", min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1)),
        ("RF", RandomForestRegressor(n_estimators=600, max_features="sqrt", min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)),
        ("HGB", HistGradientBoostingRegressor(max_iter=500, learning_rate=0.04, max_leaf_nodes=31, l2_regularization=0.02, random_state=RANDOM_STATE)),
    ]


def select_regression_rows(df_train):
    target_window = df_train[(df_train.BW > 0) & (df_train.FL >= TARGET_FL_MIN) & (df_train.FU <= TARGET_FU_MAX)].reset_index(drop=True)
    if len(target_window) >= 100:
        return target_window
    return df_train[df_train.BW > 0].reset_index(drop=True)


def fit_regressors(df_train):
    train_res = select_regression_rows(df_train)
    X_res = build_features(train_res)
    regressors = {}
    for target in ["BW", "FL", "FU"]:
        fitted = []
        for name, model in regressor_templates():
            model.fit(X_res, train_res[target].to_numpy())
            fitted.append((name, model))
        regressors[target] = fitted
    return regressors, train_res


def predict_target(regressors, X_eval, target):
    preds = np.column_stack([model.predict(X_eval) for _, model in regressors[target]])
    return preds @ REGRESSOR_WEIGHTS


def predict_target_with_std(regressors, X_eval, target):
    preds = np.column_stack([model.predict(X_eval) for _, model in regressors[target]])
    return preds @ REGRESSOR_WEIGHTS, preds.std(axis=1)

## 5. Holdout Training Run

This evaluates the base model on a held-out split before retraining on all rows for deployment.

In [6]:
idx_train, idx_val = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df.is_resonant,
)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_val = df.iloc[idx_val].reset_index(drop=True)
X_train = build_features(df_train)
X_val = build_features(df_val)
y_train = df_train.is_resonant.to_numpy()
y_val = df_val.is_resonant.to_numpy()

print("Train rows:", len(df_train), "resonant:", int(y_train.sum()), "dead:", int((y_train == 0).sum()))
print("Validation rows:", len(df_val), "resonant:", int(y_val.sum()), "dead:", int((y_val == 0).sum()))

classifier = fit_classifier(X_train, y_train)
val_prob = predict_res_prob(classifier, X_val)
val_pred = (val_prob >= classifier["threshold"]).astype(int)
val_classifier_metrics = {
    "accuracy": accuracy_score(y_val, val_pred),
    "balanced_accuracy": balanced_accuracy_score(y_val, val_pred),
    "f1": f1_score(y_val, val_pred, zero_division=0),
    "auc": roc_auc_score(y_val, val_prob),
    "dead_recall": recall_score(y_val, val_pred, pos_label=0, zero_division=0),
    "resonant_recall": recall_score(y_val, val_pred, pos_label=1, zero_division=0),
    "threshold": classifier["threshold"],
}
print(json.dumps({k: float(v) for k, v in val_classifier_metrics.items()}, indent=2))
print(confusion_matrix(y_val, val_pred, labels=[0, 1]))
print(classification_report(y_val, val_pred, labels=[0, 1], target_names=["Dead", "Resonant"], zero_division=0))

regressors, reg_train_data = fit_regressors(df_train)
val_reg_mask = (df_val.BW > 0) & (df_val.FL >= TARGET_FL_MIN) & (df_val.FU <= TARGET_FU_MAX)
if val_reg_mask.sum() < 10:
    val_reg_mask = df_val.BW > 0

regression_metrics = {}
for target in ["BW", "FL", "FU"]:
    y_true = df_val.loc[val_reg_mask, target].to_numpy()
    y_pred = predict_target(regressors, X_val.loc[val_reg_mask], target)
    regression_metrics[target] = {
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
    }
print(json.dumps({t: {k: float(v) for k, v in m.items()} for t, m in regression_metrics.items()}, indent=2))

Train rows: 1631 resonant: 1292 dead: 339
Validation rows: 408 resonant: 323 dead: 85
classifier fold 1/5 done
classifier fold 2/5 done
classifier fold 3/5 done
classifier fold 4/5 done
classifier fold 5/5 done
{
  "accuracy": 0.8848039215686274,
  "balanced_accuracy": 0.823219814241486,
  "f1": 0.9273570324574961,
  "auc": 0.9093789837916592,
  "dead_recall": 0.7176470588235294,
  "resonant_recall": 0.9287925696594427,
  "threshold": 0.4699999999999998
}
[[ 61  24]
 [ 23 300]]
              precision    recall  f1-score   support

        Dead       0.73      0.72      0.72        85
    Resonant       0.93      0.93      0.93       323

    accuracy                           0.88       408
   macro avg       0.83      0.82      0.82       408
weighted avg       0.88      0.88      0.88       408

{
  "BW": {
    "r2": 0.5596822904049199,
    "mae": 0.33148926968015935,
    "rmse": 0.5208914395859926
  },
  "FL": {
    "r2": 0.6657843655070705,
    "mae": 0.2324967871497988,
    "rmse

## 6. Retrain Deployment Model On All Data

After validation, train the deployable base model on every known simulation row. This artifact is the one downstream prediction and physics-guided notebooks should load.

In [7]:
X_full = build_features(df)
y_full = df.is_resonant.to_numpy()
final_classifier = fit_classifier(X_full, y_full)
final_regressors, final_reg_train_data = fit_regressors(df)

artifact = {
    "classifier": final_classifier,
    "regressors": final_regressors,
    "input_cols": INPUT_COLS,
    "feature_columns": FEATURE_COLUMNS,
    "regressor_weights": REGRESSOR_WEIGHTS.tolist(),
    "training_rows": int(len(df)),
    "regression_training_rows": int(len(final_reg_train_data)),
}

print("Deployment classifier threshold:", final_classifier["threshold"])
print("Deployment regression rows:", len(final_reg_train_data))

classifier fold 1/5 done
classifier fold 2/5 done
classifier fold 3/5 done
classifier fold 4/5 done
classifier fold 5/5 done
Deployment classifier threshold: 0.4699999999999998
Deployment regression rows: 1615


## 7. Rank Candidate Grid

This generates maximum-output candidates over the legal design grid. Known rows are retained in one file and excluded in another file.

In [8]:
def make_grid():
    rows = []
    wf_vals = WF_VALS_4P if HAS_LR else WF_VALS_3P
    if HAS_LR:
        for lg in LG_VALS:
            for wr in WR_VALS:
                for lr in LR_VALS:
                    for wf in wf_vals:
                        rows.append({"LG": lg, "WR": wr, "LR": lr, "WF": wf})
    else:
        for lg in LG_VALS:
            for wr in WR_VALS:
                for wf in wf_vals:
                    rows.append({"LG": lg, "WR": wr, "WF": wf})
    return pd.DataFrame(rows)


grid = make_grid()
X_grid = build_features(grid)
grid["prob_resonant"] = predict_res_prob(final_classifier, X_grid)
grid["pred_BW"], grid["bw_model_std"] = predict_target_with_std(final_regressors, X_grid, "BW")
grid["pred_FL"], grid["fl_model_std"] = predict_target_with_std(final_regressors, X_grid, "FL")
grid["pred_FU"], grid["fu_model_std"] = predict_target_with_std(final_regressors, X_grid, "FU")
grid["pred_BW"] = grid.pred_BW.clip(lower=0)
grid["pred_in_4p5_9GHz"] = ((grid.pred_FL >= TARGET_FL_MIN) & (grid.pred_FU <= TARGET_FU_MAX)).astype(int)
grid["expected_BW"] = grid.prob_resonant * grid.pred_BW
grid["confidence_penalty"] = 1.0 / (1.0 + grid.bw_model_std)
grid["selection_score"] = grid.expected_BW * grid.confidence_penalty * (0.80 + 0.20 * grid.pred_in_4p5_9GHz)

known_keys = set(tuple(row) for row in df[INPUT_COLS].round(3).to_numpy())
grid["known_in_data"] = [tuple(row) in known_keys for row in grid[INPUT_COLS].round(3).to_numpy()]

ranked_all = grid.sort_values("selection_score", ascending=False).reset_index(drop=True)
ranked_all.insert(0, "rank", np.arange(1, len(ranked_all) + 1))
ranked_unseen = ranked_all[~ranked_all.known_in_data].reset_index(drop=True)
ranked_unseen["rank"] = np.arange(1, len(ranked_unseen) + 1)

ranked_all.head(50).to_csv(PRED_DIR / "base_model_top_50_all_grid.csv", index=False)
ranked_unseen.head(50).to_csv(PRED_DIR / "base_model_top_50_unseen.csv", index=False)
ranked_unseen.head(25).to_csv(PRED_DIR / "base_model_top_25_unseen.csv", index=False)

display(ranked_unseen.head(25)[["rank"] + INPUT_COLS + ["prob_resonant", "pred_FL", "pred_FU", "pred_BW", "expected_BW", "selection_score"]])

,rank,LG,WR,LR,WF,prob_resonant,pred_FL,pred_FU,pred_BW,expected_BW,selection_score
0,1,5.5,12.25,3.0,3.5,0.543014,5.304593,8.377037,3.251746,1.765742,1.536291
1,2,5.5,12.25,3.0,3.6,0.538799,5.280271,8.302901,3.178645,1.712650,1.518291
2,3,5.5,12.25,3.0,3.4,0.544365,5.339962,8.229803,3.077781,1.675435,1.491093
3,4,6.5,12.00,3.3,3.0,0.544371,5.613674,8.334074,2.753427,1.498887,1.446531
4,5,6.5,12.25,3.3,3.5,0.544664,5.407012,8.016613,2.681193,1.460349,1.433014
5,6,6.5,12.25,3.0,3.3,0.540998,5.512959,8.133317,2.686790,1.453548,1.425343
6,7,6.0,12.25,3.3,4.6,0.545071,5.089956,8.019397,3.004178,1.637490,1.418446
7,8,6.5,12.25,3.3,3.7,0.542091,5.388007,8.079177,2.817867,1.527540,1.417532
8,9,6.0,12.25,3.0,2.7,0.540329,5.527762,8.183243,2.798622,1.512177,1.409016
9,10,6.5,12.25,3.0,3.4,0.543835,5.476558,8.118576,2.740465,1.490360,1.405379


## 8. Save Artifact And Summary

The saved pickle intentionally keeps the same top-level keys used by the current prediction workflow: `classifier`, `regressors`, and `summary`.

In [9]:
summary = {
    "goal": "train deployable base antenna model from scratch and rank maximum-output candidates",
    "input_cols": INPUT_COLS,
    "feature_count": len(FEATURE_COLUMNS),
    "rows": int(len(df)),
    "resonant": int(df.is_resonant.sum()),
    "dead": int((df.is_resonant == 0).sum()),
    "target_window_rows": int(df.in_target_window.sum()),
    "holdout_classifier": {k: float(v) for k, v in val_classifier_metrics.items()},
    "holdout_regression": {target: {k: float(v) for k, v in values.items()} for target, values in regression_metrics.items()},
    "deployment_threshold": float(final_classifier["threshold"]),
    "best_known_data_row": df.sort_values("BW", ascending=False).iloc[0][INPUT_COLS + ["FL", "FU", "BW"]].to_dict(),
    "best_predicted_all_grid_row": ranked_all.iloc[0][INPUT_COLS + ["prob_resonant", "pred_FL", "pred_FU", "pred_BW", "expected_BW", "selection_score", "known_in_data"]].to_dict(),
    "best_predicted_unseen_row": ranked_unseen.iloc[0][INPUT_COLS + ["prob_resonant", "pred_FL", "pred_FU", "pred_BW", "expected_BW", "selection_score"]].to_dict() if len(ranked_unseen) else None,
}
artifact["summary"] = summary

legacy_model_dir = PROJECT_DIR / "active_val_high_bw_resonant_outputs"
legacy_model_dir.mkdir(exist_ok=True)

model_path = MODEL_DIR / "base_antenna_model.pkl"
compat_model_path = MODEL_DIR / "high_bw_resonant_model.pkl"
legacy_compat_model_path = legacy_model_dir / "high_bw_resonant_model.pkl"
summary_path = MODEL_DIR / "base_model_summary.json"
threshold_path = MODEL_DIR / "base_model_threshold_search.csv"
metrics_path = MODEL_DIR / "base_model_classifier_metrics.csv"

with open(model_path, "wb") as f:
    pickle.dump(artifact, f)

# Compatibility artifact for existing bundle paths.
with open(compat_model_path, "wb") as f:
    pickle.dump({"classifier": final_classifier, "regressors": final_regressors, "summary": summary}, f)
with open(legacy_compat_model_path, "wb") as f:
    pickle.dump({"classifier": final_classifier, "regressors": final_regressors, "summary": summary}, f)

summary_path.write_text(json.dumps(summary, indent=2))
final_classifier["threshold_table"].to_csv(threshold_path, index=False)
pd.DataFrame([summary["holdout_classifier"]]).to_csv(metrics_path, index=False)

print("Saved:", model_path)
print("Saved compatibility model:", compat_model_path)
print("Saved legacy compatibility model:", legacy_compat_model_path)
print("Saved:", summary_path)
print("Saved:", threshold_path)
print("Saved:", metrics_path)
print(json.dumps(summary, indent=2))

Saved: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/model/base_antenna_model.pkl
Saved compatibility model: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/model/high_bw_resonant_model.pkl
Saved legacy compatibility model: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/active_val_high_bw_resonant_outputs/high_bw_resonant_model.pkl
Saved: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/model/base_model_summary.json
Saved: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/model/base_model_threshold_search.csv
Saved: /Users/adityapathania/Codes/curin/antena-model/Wideband/final-files/model/base_model_classifier_metrics.csv
{
  "goal": "train deployable base antenna model from scratch and rank maximum-output candidates",
  "input_cols": [
    "LG",
    "WR",
    "LR",
    "WF"
  ],
  "feature_count": 64,
  "rows": 2039,
  "resonant": 1615,
  "dead": 424,
  "target_window_rows": 1615,
  "ho